# 6-7 Store

In [12]:
import pandas as pd
import numpy as np
import pytz
import pulp
import random

from datetime import datetime
from tqdm import tqdm


import warnings
warnings.filterwarnings("ignore")

In [44]:
df_complete = pd.read_parquet('raw_data/df_sales_country.parquet', engine='pyarrow')

display(df_complete.head(5))
display(df_complete[['net_sales', 'gross_sales', 'qty_sold']].describe())

,dt_ref,store_id,state,sales_region,net_sales,gross_sales,qty_sold
0,2026-01-01,STORE-0067,New York,Northeast,36361.16,42056.28,4019
1,2026-01-01,STORE-0074,New York,Northeast,54537.80,66148.37,4312
2,2026-01-01,STORE-0264,Michigan,Midwest,6662.84,7734.35,567
3,2026-01-01,STORE-0307,Connecticut,Northeast,15012.83,19541.29,2383
4,2026-01-01,STORE-0384,New York,Northeast,36296.36,44760.88,3033


,net_sales,gross_sales,qty_sold
count,4.402570e+05,4.402570e+05,440257.0
mean,2.223055e+04,2.658801e+04,2820.979078
std,2.535722e+04,2.959463e+04,4454.221994
min,6.300000e-01,6.900000e-01,1.0
25%,1.065993e+04,1.278014e+04,1359.0
50%,1.596257e+04,1.910286e+04,2031.0
75%,2.550842e+04,3.052686e+04,3178.0
max,2.827760e+06,2.884773e+06,679834.0


In [42]:
def stores_optimizer(df_state: pd.DataFrame):
    df = df_state.copy()
    # 1. IDENTIFICAÇÃO DOS GRUPOS
    # Lojas que pertencem ao grupo de teste
    list_store_test = (df.loc[df['flag_test_control'] == 'test', 'store_id'].dropna().unique().tolist())
    df_store_test = pd.DataFrame({'store_id': list_store_test, 'flag': 'test'})

    # Identificação dos grupos
    df_temp_opt = df.merge(df_store_test, on='store_id', how='left')
    df_temp_opt['flag'] = df_temp_opt['flag'].fillna('control')

    # 2. CONVERSÃO DOS DADOS
    # Lojas candidatas ao grupo de controle
    store_set = (df_temp_opt.loc[df_temp_opt['flag'] == 'control', 'store_id'].dropna().unique().tolist())

    # Quantidade de lojas de teste
    n_test_stores = len(list_store_test)

    # Vendas diárias do grupo de teste
    daily_sales = (df_temp_opt.loc[df_temp_opt['flag'] == 'test'].groupby('dt_ref')['net_sales'].sum().to_dict())

    # Vendas diárias por loja candidata ao controle
    sales_store_id = (df_temp_opt.loc[df_temp_opt['flag'] == 'control'].groupby(['store_id', 'dt_ref'])['net_sales'].sum().to_dict())


    # 3. DECLARAÇÃO DO PROBLEMA
    prob = pulp.LpProblem('experiment_design', pulp.LpMinimize)

    # Variáveis binárias de seleção das lojas
    x = prob.add_variable_dicts('store_id', store_set, lowBound=0, upBound=1, cat=pulp.LpInteger)

    # Diferença percentual absoluta por dia
    sales_dif = prob.add_variable_dicts('sales_dif', list(daily_sales.keys()), lowBound=0, cat=pulp.LpContinuous)


    # 4. RESTRIÇÕES
    for day in daily_sales:
        sales_control = pulp.lpSum(x[store] * sales_store_id.get((store, day), 0) for store in store_set)

        # Normalização pelo faturamento do grupo teste
        denominator = max(abs(daily_sales[day]), 1)
        difference = (sales_control - daily_sales[day]) / denominator
        # Valor absoluto da diferença percentual
        prob += sales_dif[day] >= difference
        prob += sales_dif[day] >= -difference


    # 5. FUNÇÃO OBJETIVO
    prob += pulp.lpSum(sales_dif[day] for day in daily_sales)

    # 6. RESTRIÇÃO DE QUANTIDADE DE LOJAS
    prob += (pulp.lpSum(x[store] for store in store_set) == n_test_stores)

    # 6. RESTRIÇÕES
    # C1: Minimizar a diferença absoluta das vendas diárias
    for day in daily_sales:
        sales_control = pulp.lpSum(x[store] * sales_store_id.get((store, day), 0) for store in store_set)
        prob += (sales_dif[day] >= sales_control - daily_sales[day])
        prob += (sales_dif[day] >= daily_sales[day] - sales_control)

    # C2: Igualar a quantidade de lojas dos grupos
    prob += (pulp.lpSum(x[store] for store in store_set) == n_test_stores)


    # 7. RESOLUÇÃO
    solver = pulp.COIN_CMD(msg=1, timeLimit=EXECUTION_TIME, gapRel=0.001)
    stats = prob.solve(solver)

    # 8. EXTRAÇÃO DOS RESULTADOS

    if not stats.has_solution:
        raise RuntimeError(f'O modelo não encontrou uma solução. Status: {stats.status_str}')

    control_group_store = [store for store in store_set if x[store].value() is not None and x[store].value() > 0.5]
    control_group = pd.DataFrame({
        'store_id': control_group_store,
        'flag': 'control'
    })

    return pd.concat([control_group, df_store_test])


In [13]:
def create_test_group(df):
    test_stores = []

    all_stores = df['store_id'].unique()
    store_count = int(len(all_stores) * PERCENT_TEST_STORES)
    temp_test_stores = random.sample(sorted(all_stores), store_count)  
    temp_test_stores = pd.DataFrame(temp_test_stores, columns=['store_id'])
    temp_test_stores['flag_test_control'] = 'test'
    test_stores.append(temp_test_stores)

    if len(test_stores) == 0:
        return None

    test_stores = pd.concat(test_stores)

    df = pd.merge(df, test_stores, how='left', on='store_id')
    return df   

In [45]:
sorted(df_complete['state'].unique().tolist())

['Arizona',
 'California',
 'Colorado',
 'Connecticut',
 'Georgia',
 'Idaho',
 'Illinois',
 'Kentucky',
 'Louisiana',
 'Maine',
 'Massachusetts',
 'Michigan',
 'Minnesota',
 'Nevada',
 'New Hampshire',
 'New Jersey',
 'New York',
 'Ohio',
 'Oregon',
 'Pennsylvania',
 'South Carolina',
 'Tennessee',
 'Texas',
 'Utah',
 'Virginia',
 'Washington',
 'Wisconsin']

In [67]:
EXECUTION_TIME = 90
PERCENT_TEST_STORES = 0.20

dt_carga = datetime.now(tz = pytz.timezone('America/Chicago'))
lst_df_result = []

#states = sorted(df_complete['state'].unique().tolist())

states = ['New York']

for state in tqdm(states):
    df_state = df_complete.loc[df_complete['state'] == state]

    test_id = str(state) + '-' + str(dt_carga).split(' ')[0]


    # Creating test group
    df_state = create_test_group(df_state)

    if df_state is not None:
        # Running the optimization algorithm
        lst_df_result.append(stores_optimizer(df_state))
        

df_result = pd.concat(lst_df_result)
    

100%|██████████| 1/1 [01:30<00:00, 90.56s/it]

Welcome to the CBC MILP Solver
Version: Devel (unstable)
Build Date: Mar 23 2026
command line - C:\Users\krupc\AppData\Local\Temp\6393d418b06f45879de621ef7b5f6c30-pulp.mps -sec 90 -ratio 0.001 -timeMode elapsed -solve -printingOptions all -solution C:\Users\krupc\AppData\Local\Temp\6393d418b06f45879de621ef7b5f6c30-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1291 COLUMNS
At line 178333 RHS
At line 179620 BOUNDS
At line 179762 ENDATA
Problem MODEL has 1286 rows, 462 columns and 176438 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+08 to 90
ratioGap was changed from 0 to 0.001
Continuous objective value is 9.17046e+06 - 0.213 seconds
Cgl0004I processed model has 1285 rows, 462 columns (141 integer (141 of which binary)) and 176297 elements
Coin0011I Conflict graph built in 0.000 seconds, density: 0.033%
Cgl0015I Clique Strengthening extended 0 cliques, 0 were dominated
Cbc0045I Nauty sparseSpace 1058776 affine 175511 coeffici

In [68]:
df_result

,store_id,flag
0,STORE-0482,control
1,STORE-1221,control
2,STORE-1310,control
3,STORE-0024,control
4,STORE-0044,control
...,...,...
30,STORE-1081,test
31,STORE-1164,test
32,STORE-1288,test
33,STORE-1327,test


In [69]:
df_result = pd.merge(df_complete, df_result, how='left', on=['store_id'])

In [70]:
df_result.sample(20)

,dt_ref,store_id,state,sales_region,net_sales,gross_sales,qty_sold,flag
292131,2026-08-02,STORE-0773,Michigan,Midwest,12572.39,15500.00,1588,NaN
78221,2026-02-28,STORE-0323,Michigan,Midwest,26324.81,29918.35,2679,NaN
300944,2026-08-08,STORE-1222,Michigan,Midwest,21022.84,24767.68,2392,NaN
438629,2026-11-16,STORE-1268,Colorado,West,8244.73,10884.05,1403,NaN
954,2026-01-02,STORE-0947,Texas,South,36501.97,46240.44,4825,NaN
15019,2026-01-13,STORE-0075,New Hampshire,Northeast,24264.77,27148.49,3479,NaN
165549,2026-05-02,STORE-0758,New Jersey,Northeast,18078.04,21334.03,2407,NaN
346337,2026-09-10,STORE-1284,Washington,West,22680.82,26254.23,803,NaN
53578,2026-02-10,STORE-0124,Michigan,Midwest,24315.94,27897.18,3629,NaN
342601,2026-09-08,STORE-0422,Michigan,Midwest,16836.50,20598.13,2460,NaN


In [71]:

import pandas as pd
import numpy as np

def evaluate_test_control(df):
    """
    Avalia a similaridade entre os grupos test e control.

    Colunas esperadas:
    dt_ref, store_id, net_sales, gross_sales, qty_sold, flag
    """

    df = df.copy()

    # Garantir formato de data
    df['dt_ref'] = pd.to_datetime(df['dt_ref'])

    # Agrupar métricas por dia e grupo
    df_daily = (
        df.groupby(['dt_ref', 'flag'], as_index=False)
        .agg(
            net_sales=('net_sales', 'sum'),
            gross_sales=('gross_sales', 'sum'),
            qty_sold=('qty_sold', 'sum'),
            n_stores=('store_id', 'nunique')
        )
    )

    # Criar séries comparativas
    metrics = [
        'net_sales',
        'gross_sales',
        'qty_sold'
    ]

    comparison = {}

    for metric in metrics:

        pivot = (
            df_daily
            .pivot(
                index='dt_ref',
                columns='flag',
                values=metric
            )
            .reindex(columns=['test', 'control'])
            .fillna(0)
        )

        pivot.columns = [
            f'{metric}_test',
            f'{metric}_control'
        ]

        pivot[f'{metric}_difference'] = (
            pivot[f'{metric}_control'] -
            pivot[f'{metric}_test']
        )

        pivot[f'{metric}_abs_difference'] = (
            pivot[f'{metric}_difference'].abs()
        )

        pivot[f'{metric}_ape'] = (
            pivot[f'{metric}_abs_difference'] /
            pivot[f'{metric}_test'].abs().replace(0, np.nan)
        ) * 100

        comparison[metric] = pivot

    # Consolidar as métricas em um único dataframe
    df_comparison = pd.concat(
        comparison.values(),
        axis=1
    ).reset_index()

    # Calcular indicadores gerais
    results = []

    for metric in metrics:

        test = df_comparison[f'{metric}_test']
        control = df_comparison[f'{metric}_control']
        abs_difference = df_comparison[f'{metric}_abs_difference']

        denominator = test.abs().sum()

        results.append({
            'metric': metric,
            'MAE': abs_difference.mean(),
            'MAPE': df_comparison[f'{metric}_ape'].mean(),
            'WAPE': (
                abs_difference.sum() / denominator * 100
                if denominator != 0 else np.nan
            ),
            'Correlation': (
                test.corr(control)
                if test.nunique() > 1 and control.nunique() > 1
                else np.nan
            )
        })

    df_metrics = pd.DataFrame(results)

    return df_comparison, df_metrics


In [72]:
df_comparison, df_metrics = evaluate_test_control(df_result)

In [73]:
df_metrics

,metric,MAE,MAPE,WAPE,Correlation
0,net_sales,32414.162461,3.652986,3.389488,0.996833
1,gross_sales,35443.854174,3.534963,3.295845,0.996972
2,qty_sold,7138.831776,5.429622,5.771608,0.986821


In [74]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Net Sales',
        'Gross Sales',
        'Quantity Sold'
    ],
    vertical_spacing=0.10
)

metrics_config = [
    ('net_sales', 'Net Sales'),
    ('gross_sales', 'Gross Sales'),
    ('qty_sold', 'Quantity Sold')
]

for i, (metric, label) in enumerate(metrics_config, start=1):

    fig.add_trace(
        go.Scatter(
            x=df_comparison['dt_ref'],
            y=df_comparison[f'{metric}_test'],
            name='Test',
            legendgroup='test',
            showlegend=(i == 1),
            mode='lines',
        ),
        row=i, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df_comparison['dt_ref'],
            y=df_comparison[f'{metric}_control'],
            name='Control',
            legendgroup='control',
            showlegend=(i == 1),
            mode='lines',
        ),
        row=i, col=1
    )

fig.update_layout(
    title='Test vs Control — Daily Performance',
    template='plotly_white',
    height=850,
    hovermode='x unified'
)

fig.show()

In [75]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. IDENTIFICAR AS LOJAS DE CADA GRUPO
control_stores = df_state['store_id'].unique().tolist()
test_stores = df_state['store_id'].unique().tolist()

# 2. FILTRAR O DATAFRAME ORIGINAL
df_plot = df_state.loc[df_state['store_id'].isin(list(control_stores) + list(test_stores))].copy()
df_plot['group'] = df_plot['store_id'].map({**{store: 'Control' for store in control_stores}, **{store: 'Test' for store in test_stores}})

# 3. AGRUPAR AS VENDAS POR DIA E GRUPO
df_daily = (df_plot.groupby(['dt_ref', 'group'], as_index=False)['net_sales'].sum())
df_daily = (df_daily.pivot(index='dt_ref',columns='group',values='net_sales').fillna(0).reset_index())
df_daily = df_daily.sort_values('dt_ref')

# 4. CALCULAR AS DIFERENÇAS
df_daily['difference'] = (df_daily['Control'] - df_daily['Test'])
df_daily['difference_pct'] = (df_daily['difference'] / df_daily['Test'].replace(0, float('nan'))) * 100

# 5. CRIAR O GRÁFICO
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=(
        'Vendas diárias: Grupo Teste vs. Controle',
        'Diferença percentual entre os grupos'
    ), row_heights=[0.65, 0.35]
)

# Vendas do grupo teste & controle
fig.add_trace(go.Scatter(x=df_daily['dt_ref'], y=df_daily['Test'], name='Grupo Teste', mode='lines+markers', line=dict(width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_daily['dt_ref'], y=df_daily['Control'], name='Grupo Controle', mode='lines+markers', line=dict(width=2)), row=1, col=1)

# Diferença percentual
fig.add_trace(go.Bar(x=df_daily['dt_ref'], y=df_daily['difference_pct'], name='Diferença percentual', hovertemplate='%{y:.2f}%<extra></extra>'), row=2, col=1)

# Linha de referência em zero
fig.add_hline(y=0, line_dash='dash', line_width=1, line_color='gray', row=2, col=1)

fig.update_layout(
    title='Comparação dos grupos experimentais',
    template='plotly_white',
    hovermode='x unified',
    height=750,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

fig.update_yaxes(title_text='Vendas (R$)', tickformat=',.2f', row=1, col=1)
fig.update_yaxes(title_text='Diferença (%)', ticksuffix='%', row=2, col=1)
fig.update_xaxes(title_text='Data', row=2, col=1)
fig.show()


KeyError: 'Control'